In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
ann = pd.read_csv("../data/processed/mhs_main_experiment_annotations_with_split.csv")

print("Loaded:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())
print(ann["split"].value_counts())

Loaded: (49433, 53)
Unique comments: 19761
split
train         34413
test           7713
validation     7307
Name: count, dtype: int64


In [3]:
women_ann = ann[ann["is_women_targeted"] == 1].copy()

print("Women-targeted annotation rows:", women_ann.shape[0])
print("Women-targeted unique comments:", women_ann["comment_id"].nunique())

Women-targeted annotation rows: 27889
Women-targeted unique comments: 11983


In [4]:
comment_counts = (
    women_ann.groupby("comment_id")
    .agg(
        split=("split", "first"),
        text_clean=("text_clean", "first"),
        total_annotations=("annotator_id", "count"),
        women_annotations=("annotator_gender_group", lambda x: (x == "women").sum()),
        men_annotations=("annotator_gender_group", lambda x: (x == "men").sum()),
        non_binary_annotations=("annotator_gender_group", lambda x: (x == "non_binary").sum()),
        prefer_not_to_say_annotations=("annotator_gender_group", lambda x: (x == "prefer_not_to_say").sum()),
        self_describe_annotations=("annotator_gender_group", lambda x: (x == "self_describe").sum())
    )
    .reset_index()
)

comment_counts.head()

,comment_id,split,text_clean,total_annotations,women_annotations,men_annotations,non_binary_annotations,prefer_not_to_say_annotations,self_describe_annotations
0,6,train,First off you look cool as fuck! Anyway if we ...,2,1,1,0,0,0
1,11,validation,"eat my fuck, bitch",2,1,1,0,0,0
2,12,train,She ugly anyways,2,2,0,0,0,0
3,19,train,Won't happen. Birth rates are down. Women want...,2,0,2,0,0,0
4,22,train,The guys are one thing but then you have a wom...,2,0,2,0,0,0


In [5]:
count_summary = comment_counts[
    [
        "total_annotations",
        "women_annotations",
        "men_annotations",
        "non_binary_annotations",
        "prefer_not_to_say_annotations",
        "self_describe_annotations"
    ]
].describe()

count_summary

,total_annotations,women_annotations,men_annotations,non_binary_annotations,prefer_not_to_say_annotations,self_describe_annotations
count,11983.000000,11983.000000,11983.000000,11983.000000,11983.000000,11983.000000
mean,2.327380,1.310440,0.989235,0.017441,0.007678,0.002587
std,15.052045,8.755565,6.191982,0.180264,0.092848,0.050799
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.000000,1.000000,1.000000,0.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000,0.000000,0.000000,0.000000
max,737.000000,422.000000,315.000000,9.000000,3.000000,1.000000


In [6]:
threshold_rows = []

for threshold in [1, 2, 3, 4, 5]:
    threshold_rows.append({
        "threshold": threshold,
        "comments_with_total_at_least_threshold": (comment_counts["total_annotations"] >= threshold).sum(),
        "comments_with_women_at_least_threshold": (comment_counts["women_annotations"] >= threshold).sum(),
        "comments_with_men_at_least_threshold": (comment_counts["men_annotations"] >= threshold).sum(),
        "comments_with_both_women_and_men_at_least_threshold": (
            (comment_counts["women_annotations"] >= threshold) &
            (comment_counts["men_annotations"] >= threshold)
        ).sum(),
        "comments_with_either_women_or_men_at_least_threshold": (
            (comment_counts["women_annotations"] >= threshold) |
            (comment_counts["men_annotations"] >= threshold)
        ).sum()
    })

threshold_audit = pd.DataFrame(threshold_rows)
threshold_audit

,threshold,comments_with_total_at_least_threshold,comments_with_women_at_least_threshold,comments_with_men_at_least_threshold,comments_with_both_women_and_men_at_least_threshold,comments_with_either_women_or_men_at_least_threshold
0,1,11983,8630,7168,3869,11929
1,2,6394,2935,1900,271,4564
2,3,2790,641,343,24,960
3,4,739,97,51,24,124
4,5,46,27,22,21,28


In [7]:
split_threshold_rows = []

for split in ["train", "validation", "test"]:
    split_df = comment_counts[comment_counts["split"] == split]

    for threshold in [1, 2, 3, 4, 5]:
        split_threshold_rows.append({
            "split": split,
            "threshold": threshold,
            "total_comments": len(split_df),
            "women_at_least_threshold": (split_df["women_annotations"] >= threshold).sum(),
            "men_at_least_threshold": (split_df["men_annotations"] >= threshold).sum(),
            "both_women_and_men_at_least_threshold": (
                (split_df["women_annotations"] >= threshold) &
                (split_df["men_annotations"] >= threshold)
            ).sum()
        })

split_threshold_audit = pd.DataFrame(split_threshold_rows)
split_threshold_audit

,split,threshold,total_comments,women_at_least_threshold,men_at_least_threshold,both_women_and_men_at_least_threshold
0,train,1,8387,6021,5020,2694
1,train,2,8387,2059,1333,186
2,train,3,8387,439,237,15
3,train,4,8387,65,33,15
4,train,5,8387,18,13,13
5,validation,1,1797,1329,1061,600
6,validation,2,1797,440,251,32
7,validation,3,1797,98,53,4
8,validation,4,1797,13,11,4
9,validation,5,1797,4,5,4


In [8]:
def reliability_category(row):
    if row["women_annotations"] >= 3 and row["men_annotations"] >= 3:
        return "strong_both"
    elif row["women_annotations"] >= 2 and row["men_annotations"] >= 2:
        return "usable_both"
    elif row["women_annotations"] >= 2:
        return "usable_women_only"
    elif row["men_annotations"] >= 2:
        return "usable_men_only"
    else:
        return "weak_or_single_annotation"

comment_counts["reliability_category"] = comment_counts.apply(reliability_category, axis=1)

reliability_summary = (
    comment_counts["reliability_category"]
    .value_counts()
    .reset_index()
)

reliability_summary.columns = ["reliability_category", "unique_comments"]

reliability_summary

,reliability_category,unique_comments
0,weak_or_single_annotation,7419
1,usable_women_only,2664
2,usable_men_only,1629
3,usable_both,247
4,strong_both,24


In [9]:
reliability_by_split = (
    comment_counts
    .groupby(["split", "reliability_category"])
    .size()
    .reset_index(name="unique_comments")
    .sort_values(["split", "unique_comments"], ascending=[True, False])
)

reliability_by_split

,split,reliability_category,unique_comments
4,test,weak_or_single_annotation,1100
3,test,usable_women_only,383
2,test,usable_men_only,263
1,test,usable_both,48
0,test,strong_both,5
9,train,weak_or_single_annotation,5181
8,train,usable_women_only,1873
7,train,usable_men_only,1147
6,train,usable_both,171
5,train,strong_both,15


In [10]:
os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/perspective_dataset_quality_audit.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("PERSPECTIVE DATASET QUALITY AUDIT\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. PURPOSE\n")
    f.write("-" * 80 + "\n")
    f.write("This audit checks whether women/men subgroup distributions are reliable enough for perspective-aware modelling.\n")
    f.write("The main concern is that subgroup distributions based on only one annotator are noisy and should not be treated as robust disagreement distributions.\n\n")

    f.write("2. DATASET SIZE\n")
    f.write("-" * 80 + "\n")
    f.write(f"Women-targeted annotation rows: {len(women_ann)}\n")
    f.write(f"Women-targeted unique comments: {women_ann['comment_id'].nunique()}\n\n")

    f.write("3. ANNOTATION COUNT SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(count_summary.to_string())
    f.write("\n\n")

    f.write("4. THRESHOLD AUDIT\n")
    f.write("-" * 80 + "\n")
    f.write(threshold_audit.to_string(index=False))
    f.write("\n\n")

    f.write("5. THRESHOLD AUDIT BY SPLIT\n")
    f.write("-" * 80 + "\n")
    f.write(split_threshold_audit.to_string(index=False))
    f.write("\n\n")

    f.write("6. RELIABILITY CATEGORY SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(reliability_summary.to_string(index=False))
    f.write("\n\n")

    f.write("7. RELIABILITY CATEGORY BY SPLIT\n")
    f.write("-" * 80 + "\n")
    f.write(reliability_by_split.to_string(index=False))
    f.write("\n\n")

   

In [11]:
immigrant_ann = ann[ann["is_immigrant_targeted"] == 1].copy()

print("Immigrant-targeted annotation rows:", immigrant_ann.shape[0])
print("Immigrant-targeted unique comments:", immigrant_ann["comment_id"].nunique())
print(immigrant_ann["split"].value_counts())

Immigrant-targeted annotation rows: 22528
Immigrant-targeted unique comments: 8621
split
train         15911
validation     3552
test           3065
Name: count, dtype: int64


In [13]:
ideology_groups = [
    "extremely_conservative",
    "conservative",
    "slightly_conservative",
    "neutral",
    "slightly_liberal",
    "liberal",
    "extremely_liberal",
    "no_opinion"
   
]

ideology_count_dict = {}

for group in ideology_groups:
    ideology_count_dict[f"{group}_annotations"] = (
        "annotator_ideology_group",
        lambda x, group=group: (x == group).sum()
    )

ideology_comment_counts = (
    immigrant_ann.groupby("comment_id")
    .agg(
        split=("split", "first"),
        text_clean=("text_clean", "first"),
        total_annotations=("annotator_id", "count"),
        **ideology_count_dict
    )
    .reset_index()
)

ideology_comment_counts.head()

,comment_id,split,text_clean,total_annotations,extremely_conservative_annotations,conservative_annotations,slightly_conservative_annotations,neutral_annotations,slightly_liberal_annotations,liberal_annotations,extremely_liberal_annotations,no_opinion_annotations
0,7,test,\*points to posters asking for palestinian rig...,2,0,0,0,0,0,1,1,0
1,10,train,"They'll come back in your plan, also. Plus we ...",3,0,0,1,1,0,1,0,0
2,13,train,Do they realize that a random Japanese person ...,2,0,0,0,0,0,1,1,0
3,23,validation,"Build the wall, and put these spics in prison ...",1,0,0,1,0,0,0,0,0
4,24,train,Love from an Indian from USA.,1,0,1,0,0,0,0,0,0


In [14]:
ideology_annotation_cols = [
    f"{group}_annotations"
    for group in ideology_groups
]

ideology_count_summary = ideology_comment_counts[
    ["total_annotations"] + ideology_annotation_cols
].describe()

ideology_count_summary

,total_annotations,extremely_conservative_annotations,conservative_annotations,slightly_conservative_annotations,neutral_annotations,slightly_liberal_annotations,liberal_annotations,extremely_liberal_annotations,no_opinion_annotations
count,8621.000000,8621.000000,8621.000000,8621.000000,8621.000000,8621.000000,8621.000000,8621.000000,8621.000000
mean,2.613154,0.088389,0.290222,0.289758,0.439624,0.417817,0.656420,0.357035,0.072961
std,18.498833,0.697913,2.003531,2.095133,3.147124,2.995957,4.746379,2.715792,0.590441
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000
max,702.000000,27.000000,85.000000,85.000000,119.000000,118.000000,180.000000,114.000000,25.000000


In [15]:
ideology_threshold_rows = []

for threshold in [1, 2, 3, 4, 5]:
    row = {
        "threshold": threshold,
        "comments_with_total_at_least_threshold": (
            ideology_comment_counts["total_annotations"] >= threshold
        ).sum()
    }

    for group in ideology_groups:
        col = f"{group}_annotations"
        row[f"{group}_at_least_threshold"] = (
            ideology_comment_counts[col] >= threshold
        ).sum()

    ideology_threshold_rows.append(row)

ideology_threshold_audit = pd.DataFrame(ideology_threshold_rows)

ideology_threshold_audit

,threshold,comments_with_total_at_least_threshold,extremely_conservative_at_least_threshold,conservative_at_least_threshold,slightly_conservative_at_least_threshold,neutral_at_least_threshold,slightly_liberal_at_least_threshold,liberal_at_least_threshold,extremely_liberal_at_least_threshold,no_opinion_at_least_threshold
0,1,8621,518,1608,1598,2325,2220,3297,1909,435
1,2,4257,43,132,113,286,275,493,208,35
2,3,1787,25,36,37,44,45,77,41,22
3,4,469,22,32,34,33,31,39,31,17
4,5,49,20,31,32,32,31,36,27,16


In [16]:
ideology_split_threshold_rows = []

for split in ["train", "validation", "test"]:
    split_df = ideology_comment_counts[
        ideology_comment_counts["split"] == split
    ]

    for threshold in [1, 2, 3, 4, 5]:
        row = {
            "split": split,
            "threshold": threshold,
            "total_comments": len(split_df),
            "comments_with_total_at_least_threshold": (
                split_df["total_annotations"] >= threshold
            ).sum()
        }

        for group in ideology_groups:
            col = f"{group}_annotations"
            row[f"{group}_at_least_threshold"] = (
                split_df[col] >= threshold
            ).sum()

        ideology_split_threshold_rows.append(row)

ideology_split_threshold_audit = pd.DataFrame(ideology_split_threshold_rows)

ideology_split_threshold_audit

,split,threshold,total_comments,comments_with_total_at_least_threshold,extremely_conservative_at_least_threshold,conservative_at_least_threshold,slightly_conservative_at_least_threshold,neutral_at_least_threshold,slightly_liberal_at_least_threshold,liberal_at_least_threshold,extremely_liberal_at_least_threshold,no_opinion_at_least_threshold
0,train,1,6031,6031,344,1124,1122,1584,1595,2320,1336,306
1,train,2,6031,2974,26,82,75,201,196,346,151,23
2,train,3,6031,1246,19,27,28,31,32,54,30,16
3,train,4,6031,324,16,23,26,24,22,26,23,14
4,train,5,6031,34,15,23,25,23,22,26,19,13
5,validation,1,1296,1296,93,247,249,364,302,492,286,59
6,validation,2,1296,641,13,29,21,45,42,75,31,7
7,validation,3,1296,271,4,7,6,8,8,14,8,4
8,validation,4,1296,77,4,7,6,6,7,8,6,2
9,validation,5,1296,8,3,6,5,6,7,7,6,2


In [17]:
def ideology_reliability_category(row):
    groups_with_2plus = 0
    groups_with_3plus = 0

    for group in ideology_groups:
        col = f"{group}_annotations"

        if row[col] >= 2:
            groups_with_2plus += 1

        if row[col] >= 3:
            groups_with_3plus += 1

    if groups_with_3plus >= 2:
        return "strong_multiple_ideology_groups"
    elif groups_with_2plus >= 2:
        return "usable_multiple_ideology_groups"
    elif groups_with_2plus == 1:
        return "usable_single_ideology_group"
    else:
        return "weak_or_single_annotation"

ideology_comment_counts["ideology_reliability_category"] = (
    ideology_comment_counts.apply(ideology_reliability_category, axis=1)
)

ideology_reliability_summary = (
    ideology_comment_counts["ideology_reliability_category"]
    .value_counts()
    .reset_index()
)

ideology_reliability_summary.columns = [
    "ideology_reliability_category",
    "unique_comments"
]

ideology_reliability_summary

,ideology_reliability_category,unique_comments
0,weak_or_single_annotation,7310
1,usable_single_ideology_group,1239
2,strong_multiple_ideology_groups,38
3,usable_multiple_ideology_groups,34


In [18]:
ideology_reliability_by_split = (
    ideology_comment_counts
    .groupby(["split", "ideology_reliability_category"])
    .size()
    .reset_index(name="unique_comments")
    .sort_values(["split", "unique_comments"], ascending=[True, False])
)

ideology_reliability_by_split

,split,ideology_reliability_category,unique_comments
3,test,weak_or_single_annotation,1098
2,test,usable_single_ideology_group,186
1,test,usable_multiple_ideology_groups,6
0,test,strong_multiple_ideology_groups,4
7,train,weak_or_single_annotation,5131
6,train,usable_single_ideology_group,850
4,train,strong_multiple_ideology_groups,26
5,train,usable_multiple_ideology_groups,24
11,validation,weak_or_single_annotation,1081
10,validation,usable_single_ideology_group,203


In [19]:
with open(report_path, "a", encoding="utf-8") as f:
    f.write("\n\n")
    f.write("IDEOLOGY PERSPECTIVE DATASET QUALITY AUDIT\n")
    f.write("=" * 80 + "\n\n")

    f.write("1. PURPOSE\n")
    f.write("-" * 80 + "\n")
    f.write("This section checks whether ideology subgroup distributions are reliable enough for immigrant-targeted perspective-aware modelling.\n")
    f.write("The main concern is whether each ideology group has enough annotations per comment to form stable subgroup distributions.\n\n")

    f.write("2. DATASET SIZE\n")
    f.write("-" * 80 + "\n")
    f.write(f"Immigrant-targeted annotation rows: {len(immigrant_ann)}\n")
    f.write(f"Immigrant-targeted unique comments: {immigrant_ann['comment_id'].nunique()}\n\n")

    f.write("3. IDEOLOGY ANNOTATION COUNT SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(ideology_count_summary.to_string())
    f.write("\n\n")

    f.write("4. IDEOLOGY THRESHOLD AUDIT\n")
    f.write("-" * 80 + "\n")
    f.write(ideology_threshold_audit.to_string(index=False))
    f.write("\n\n")

    f.write("5. IDEOLOGY THRESHOLD AUDIT BY SPLIT\n")
    f.write("-" * 80 + "\n")
    f.write(ideology_split_threshold_audit.to_string(index=False))
    f.write("\n\n")

    f.write("6. IDEOLOGY RELIABILITY CATEGORY SUMMARY\n")
    f.write("-" * 80 + "\n")
    f.write(ideology_reliability_summary.to_string(index=False))
    f.write("\n\n")

    f.write("7. IDEOLOGY RELIABILITY CATEGORY BY SPLIT\n")
    f.write("-" * 80 + "\n")
    f.write(ideology_reliability_by_split.to_string(index=False))
    f.write("\n\n")

    f.write("8. INTERPRETATION NOTES\n")
    f.write("-" * 80 + "\n")
    f.write("If most ideology subgroup distributions are based on only one annotator, ideology-conditioned modelling will be noisy.\n")
    f.write("A threshold of at least 2 annotations per ideology group is a reasonable minimum for subgroup distribution training.\n")
    f.write("If very few comments contain multiple ideology groups with at least 2 annotations each, direct ideology-perspective prediction may not be feasible.\n")